# Foody Dataset Cleaning Notebook

Pipeline lam sach du lieu crawl tu Foody.vn, chuan bi cho bai toan **Multimodal AI Regression**
(du doan `avg_rating` tu Review Text + Review Image).

### Pipeline phases

| Phase | Muc tieu |
|:-----:|----------|
| 1 - Technical Cleaning | Sua loi encoding/HTML entity, chuan hoa Unicode, datetime, kieu so, dedup |
| 2 - Content Cleaning | Danh dau (khong xoa) spam/quang cao, review qua ngan, dem emoji |
| 3 - ML Cleaning | Loc review hop le, dung dataset multimodal va text-only de train |

### Input (khong sua / khong ghi de)
```
/content/drive/MyDrive/Scrawl_data/Foody/checkpoints/
    restaurants_checkpoint.csv
    reviews_checkpoint.csv
    review_images_checkpoint.csv
```

### Output
```
/content/drive/MyDrive/Scrawl_data/Foody/checkpoints_clean/
    restaurants_clean.csv (+ .json)
    reviews_clean.csv (+ .json)
    review_images_clean.csv (+ .json)
    text_only_reviews.csv
    multimodal_reviews.csv
    cleaning_report.json
```

### Luu y quan trong
- Bai toan la **Regression** — target la `avg_rating` (gia tri lien tuc), **KHONG** phai Classification.
- **Khong** tao `rating_label`, **khong** chuyen rating thanh positive/negative/neutral.
- **Khong** xoa cot `comment` goc — moi bien doi noi dung duoc luu trong cot moi `comment_clean`.
- **Khong** sua/ghi de file raw trong `checkpoints/` — toan bo output ghi vao `checkpoints_clean/`.
- **Khong** download anh — chi xu ly metadata/URL va giu nguyen mapping review-image.
- Spam / review qua ngan chi duoc **danh dau** o Phase 2, va **loc** o Phase 3 (ML Cleaning).


## 1. Setup

Import thu vien va mount Google Drive (chay tren Google Colab).


In [ ]:
import os
import re
import json
import html
import logging
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

# Mount Google Drive khi chay tren Colab — bo qua neu chay local
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ImportError:
    print('Khong chay tren Colab — bo qua drive.mount().')

print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print('Import thanh cong.')


## 2. Configuration

Cau hinh duong dan input/output va cac nguong (threshold) dung trong Phase 2 / Phase 3.
`checkpoints_clean/` nam ngang cap voi `checkpoints/` — khong ghi de du lieu goc.


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR  = "/content/drive/MyDrive/Scrawl_data/Foody"
RAW_DIR   = os.path.join(BASE_DIR, "checkpoints")
CLEAN_DIR = os.path.join(BASE_DIR, "checkpoints_clean")

RESTAURANTS_FILE   = os.path.join(RAW_DIR, "restaurants_checkpoint.csv")
REVIEWS_FILE       = os.path.join(RAW_DIR, "reviews_checkpoint.csv")
REVIEW_IMAGES_FILE = os.path.join(RAW_DIR, "review_images_checkpoint.csv")

# Thu muc output rieng — khong dung lai ten file/thu muc cua raw data
os.makedirs(CLEAN_DIR, exist_ok=True)

# ── Nguong dung trong Phase 2 (Content) / Phase 3 (ML) ──────────────────────
MIN_COMMENT_LENGTH = 15     # so ky tu toi thieu — duoi muc nay -> is_too_short = True
MIN_WORD_COUNT     = 3      # so tu toi thieu  — duoi muc nay -> is_too_short = True
MIN_VALID_RATING   = 0.0    # avg_rating hop le phai nam trong [MIN, MAX]
MAX_VALID_RATING   = 10.0

# Tu khoa quang cao / spam tieng Viet — chi dung de DANH DAU, khong dung de xoa truc tiep
SPAM_KEYWORDS: List[str] = [
    'voucher', 'khuyến mãi', 'khuyen mai', 'ưu đãi', 'uu dai',
    'giảm giá', 'giam gia', 'mua 2 tặng 1', 'mua 1 tặng 1',
    'số lượng có hạn', 'so luong co han', 'truy cập', 'truy cap',
    'deal', 'promo', 'freeship', 'inbox', 'order ngay', 'đặt ngay',
]

# Regex loai bo URL / link rut gon trong comment
URL_PATTERN = re.compile(r'(https?://\S+|www\.\S+|shorten\.asia\S*)', re.IGNORECASE)

# Regex nhan dien emoji (cac khoi Unicode pho bien)
EMOJI_PATTERN = re.compile(
    '['
    '\U0001F300-\U0001FAFF'   # pictographs, emoticons, transport, supplemental symbols
    '\U00002600-\U000027BF'   # misc symbols & dingbats
    '\U0001F1E6-\U0001F1FF'   # regional indicator symbols (co quoc gia)
    '\U00002B00-\U00002BFF'   # misc symbols and arrows
    '\U0001F000-\U0001F0FF'   # mahjong / playing cards / dominoes
    ']',
    flags=re.UNICODE
)

print('Da nap cau hinh.')
print(f'  RAW_DIR   : {RAW_DIR}')
print(f'  CLEAN_DIR : {CLEAN_DIR}')
print(f'  Restaurants file   : {RESTAURANTS_FILE}')
print(f'  Reviews file       : {REVIEWS_FILE}')
print(f'  Review images file : {REVIEW_IMAGES_FILE}')
print(f'  MIN_COMMENT_LENGTH = {MIN_COMMENT_LENGTH}, MIN_WORD_COUNT = {MIN_WORD_COUNT}')
print(f'  Khoang avg_rating hop le = [{MIN_VALID_RATING}, {MAX_VALID_RATING}]')


## 3. Load Raw Checkpoint Data

Doc 3 file checkpoint goc (chi doc — khong ghi/sua). Neu thieu file, bao loi ro rang
va tra ve DataFrame rong de cac cell phia sau van chay an toan thay vi crash mo ho.


In [ ]:
def load_checkpoint_csv(path: str, label: str) -> pd.DataFrame:
    """Doc mot file checkpoint CSV; bao loi ro rang neu file khong ton tai."""
    if not os.path.exists(path):
        logger.error(f'{label}: khong tim thay file -> {path}')
        print(f'LOI: Khong tim thay file {label} tai:')
        print(f'  {path}')
        print('  -> Kiem tra lai BASE_DIR va dam bao crawler da tao checkpoint.')
        return pd.DataFrame()
    df = pd.read_csv(path, encoding='utf-8-sig')
    logger.info(f'{label}: nap {len(df):,} dong x {len(df.columns)} cot tu {path}')
    return df


df_restaurants_raw   = load_checkpoint_csv(RESTAURANTS_FILE, 'Restaurants')
df_reviews_raw       = load_checkpoint_csv(REVIEWS_FILE, 'Reviews')
df_review_images_raw = load_checkpoint_csv(REVIEW_IMAGES_FILE, 'Review images')

print()
print('Kich thuoc du lieu raw:')
print(f'  df_restaurants_raw   : {df_restaurants_raw.shape}')
print(f'  df_reviews_raw       : {df_reviews_raw.shape}')
print(f'  df_review_images_raw : {df_review_images_raw.shape}')

if df_reviews_raw.empty:
    print()
    print('CANH BAO: df_reviews_raw rong — cac phase lam sach phia sau se khong co gi de xu ly.')


## 4. Initial Data Audit

Khao sat nhanh du lieu tho: schema, kieu du lieu, ty le thieu, va mau du lieu loi
(HTML entity, dinh dang `/Date(...)/`, v.v.) — lam co so doi chieu truoc/sau khi clean.


In [ ]:
def audit_dataframe(df: pd.DataFrame, name: str, sample_cols: Optional[List[str]] = None) -> None:
    """In nhanh thong tin audit: shape, missing values, va vai gia tri raw mau."""
    print('=' * 65)
    print(f'  AUDIT: {name}')
    print('=' * 65)
    if df.empty:
        print('  (DataFrame rong — khong co gi de audit)')
        return

    print(f'Shape  : {df.shape[0]:,} dong x {df.shape[1]} cot')
    print(f'Columns: {list(df.columns)}')
    print()
    print('Missing values (top 10):')
    missing = df.isnull().sum().sort_values(ascending=False)
    missing = missing[missing > 0].head(10)
    if not missing.empty:
        print(missing.to_string())
    else:
        print('  (khong co gia tri thieu)')

    if sample_cols:
        present = [c for c in sample_cols if c in df.columns]
        if present:
            print()
            print(f'Mau gia tri raw ({", ".join(present)}):')
            for col in present:
                sample_vals = df[col].dropna().astype(str).head(3).tolist()
                print(f'  {col}: {sample_vals}')
    print()


audit_dataframe(df_restaurants_raw, 'restaurants_checkpoint (raw)',
                sample_cols=['restaurant_name', 'address'])
audit_dataframe(df_reviews_raw, 'reviews_checkpoint (raw)',
                sample_cols=['comment', 'created_on', 'updated_on', 'avg_rating'])
audit_dataframe(df_review_images_raw, 'review_images_checkpoint (raw)',
                sample_cols=['image_url', 'width', 'height'])


## 5. Phase 1 — Technical Cleaning

**Muc tieu:** sua loi ky thuat de du lieu doc dung — KHONG danh gia chat luong noi dung,
KHONG drop manh tay (chi loai bo cac dong thuc su khong dung duoc, vi du thieu khoa chinh).

Cac buoc:
1. Giai ma HTML entity (`html.unescape`) tren cac cot text
2. Chuan hoa Unicode ve dang NFC (`unicodedata.normalize`)
3. Chuan hoa khoang trang / tab / xuong dong
4. Chuyen dinh dang `/Date(epoch_ms)/` -> `datetime` chuan, tach them `created_year`, `created_month`
5. Ep kieu numeric cho cac cot diem so / dem luot
6. Chuan hoa missing values (chuoi rong, "null", "None", NaN)
7. Loai ban ghi trung lap theo khoa chinh (`restaurant_id`, `review_id`, `image_id`)


In [ ]:
# ── Cac ham xu ly cho Phase 1 ────────────────────────────────────────────────

def decode_html_entities(text: Any) -> Any:
    """Giai ma HTML entity (vd: m&#236;nh -> mình) va chuan hoa Unicode ve NFC."""
    if not isinstance(text, str):
        return text
    text = html.unescape(text)
    text = unicodedata.normalize('NFC', text)
    return text


def normalize_whitespace(text: Any) -> Any:
    """Gop tab/xuong dong/nhieu khoang trang lien tiep thanh 1 dau cach; strip 2 dau."""
    if not isinstance(text, str):
        return text
    text = text.replace('\r', ' ').replace('\n', ' ').replace('\t', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def technical_clean_text(text: Any) -> Any:
    """Pipeline lam sach ky thuat cho text: HTML decode -> Unicode NFC -> chuan hoa whitespace."""
    text = decode_html_entities(text)
    text = normalize_whitespace(text)
    return text


def parse_foody_date(value: Any) -> Optional[pd.Timestamp]:
    """Parse dinh dang '/Date(epoch_ms)/' cua Foody (va ca chuoi ISO thuong) -> Timestamp UTC."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (int, float)):
        try:
            return pd.to_datetime(int(value), unit='ms', utc=True)
        except (ValueError, OverflowError):
            return None

    text = str(value).strip()
    if not text or text.lower() in ('nan', 'none', 'null'):
        return None

    match = re.match(r'/Date\((-?\d+)([+-]\d{4})?\)/', text)
    if match:
        epoch_ms = int(match.group(1))
        try:
            return pd.to_datetime(epoch_ms, unit='ms', utc=True)
        except (ValueError, OverflowError):
            return None

    parsed = pd.to_datetime(text, errors='coerce', utc=True)
    return parsed if pd.notna(parsed) else None


def normalize_missing(value: Any, fill: Any = None) -> Any:
    """Quy cac bien the 'thieu du lieu' (chuoi rong / 'null' / 'none' / NaN) ve cung mot gia tri fill."""
    if value is None:
        return fill
    if isinstance(value, float) and pd.isna(value):
        return fill
    if isinstance(value, str) and value.strip().lower() in ('', 'null', 'none', 'nan', 'n/a'):
        return fill
    return value


print('Da dinh nghia ham Phase 1: decode_html_entities, normalize_whitespace,')
print('  technical_clean_text, parse_foody_date, normalize_missing')


In [ ]:
# ── Technical cleaning: restaurants ──────────────────────────────────────────
df_restaurants_tech = df_restaurants_raw.copy()

if not df_restaurants_tech.empty:
    _restaurant_text_cols = ['restaurant_name', 'address', 'restaurant_url', 'restaurant_full_url']
    for col in _restaurant_text_cols:
        if col in df_restaurants_tech.columns:
            df_restaurants_tech[col] = df_restaurants_tech[col].apply(technical_clean_text)

    # Chuan hoa missing values cho cac truong text quan trong
    for col in ['restaurant_name', 'address']:
        if col in df_restaurants_tech.columns:
            df_restaurants_tech[col] = df_restaurants_tech[col].apply(lambda v: normalize_missing(v, fill='unknown'))

    # Ep kieu numeric
    for col in ['avg_rating', 'total_reviews', 'latitude', 'longitude']:
        if col in df_restaurants_tech.columns:
            df_restaurants_tech[col] = pd.to_numeric(df_restaurants_tech[col], errors='coerce')

    # Datetime
    if 'crawl_timestamp' in df_restaurants_tech.columns:
        df_restaurants_tech['crawl_timestamp'] = pd.to_datetime(
            df_restaurants_tech['crawl_timestamp'], errors='coerce', utc=True
        )

    # Loai trung lap theo restaurant_id (giu ban ghi dau tien)
    _before = len(df_restaurants_tech)
    df_restaurants_tech.drop_duplicates(subset=['restaurant_id'], keep='first', inplace=True)
    df_restaurants_tech.reset_index(drop=True, inplace=True)
    _removed_dup_restaurants = _before - len(df_restaurants_tech)
    logger.info(f'Restaurants dedup: {_before} -> {len(df_restaurants_tech)} (loai {_removed_dup_restaurants})')
else:
    _removed_dup_restaurants = 0

print(f'df_restaurants_tech shape: {df_restaurants_tech.shape}')
print(f'So restaurant trung lap da loai: {_removed_dup_restaurants}')


In [ ]:
# ── Technical cleaning: reviews ───────────────────────────────────────────────
df_reviews_tech = df_reviews_raw.copy()

if not df_reviews_tech.empty:
    _review_text_cols = [
        'restaurant_name', 'user_name', 'title', 'comment',
        'device_name', 'review_type_name', 'created_date_raw',
        'review_url', 'source_restaurant_url',
    ]
    for col in _review_text_cols:
        if col in df_reviews_tech.columns:
            df_reviews_tech[col] = df_reviews_tech[col].apply(technical_clean_text)

    # Chuan hoa missing values — GIU NGUYEN cot 'comment' goc, chi normalize placeholder rong/null
    if 'user_name' in df_reviews_tech.columns:
        df_reviews_tech['user_name'] = df_reviews_tech['user_name'].apply(
            lambda v: normalize_missing(v, fill='unknown_user')
        )
    if 'comment' in df_reviews_tech.columns:
        df_reviews_tech['comment'] = df_reviews_tech['comment'].apply(lambda v: normalize_missing(v, fill=''))
    if 'title' in df_reviews_tech.columns:
        df_reviews_tech['title'] = df_reviews_tech['title'].apply(lambda v: normalize_missing(v, fill=''))

    # Chuyen '/Date(epoch_ms)/' (hoac chuoi ISO) -> datetime chuan + tach nam/thang
    for src_col, dt_col in [('created_on', 'created_datetime'), ('updated_on', 'updated_datetime')]:
        if src_col in df_reviews_tech.columns:
            df_reviews_tech[dt_col] = df_reviews_tech[src_col].apply(parse_foody_date)
        else:
            df_reviews_tech[dt_col] = pd.NaT

    df_reviews_tech['created_year']  = df_reviews_tech['created_datetime'].dt.year
    df_reviews_tech['created_month'] = df_reviews_tech['created_datetime'].dt.month

    # Ep kieu numeric cho cac cot diem so / id / bo dem
    _numeric_cols = [
        'avg_rating', 'food_score', 'service_score', 'atmosphere_score',
        'position_score', 'price_score', 'total_views', 'total_like',
        'total_comment', 'image_count', 'device_type', 'review_id',
        'restaurant_id', 'user_id', 'review_type',
    ]
    for col in _numeric_cols:
        if col in df_reviews_tech.columns:
            df_reviews_tech[col] = pd.to_numeric(df_reviews_tech[col], errors='coerce')

    # Cac cot dem luot tuong tac: thieu -> hieu la "khong co tuong tac" -> dien 0
    for col in ['total_views', 'total_like', 'total_comment', 'image_count']:
        if col in df_reviews_tech.columns:
            df_reviews_tech[col] = df_reviews_tech[col].fillna(0).astype('Int64')

    # Loai cac dong khong co review_id (khong the dinh danh / join)
    _before = len(df_reviews_tech)
    df_reviews_tech.dropna(subset=['review_id'], inplace=True)
    _dropped_no_id = _before - len(df_reviews_tech)

    # Loai trung lap theo review_id (giu ban ghi dau tien)
    _before = len(df_reviews_tech)
    df_reviews_tech.drop_duplicates(subset=['review_id'], keep='first', inplace=True)
    df_reviews_tech.reset_index(drop=True, inplace=True)
    _removed_dup_reviews = _before - len(df_reviews_tech)

    logger.info(f'Reviews: loai {_dropped_no_id} dong khong co review_id, '
                f'loai {_removed_dup_reviews} dong trung lap -> con {len(df_reviews_tech):,} dong')
else:
    _dropped_no_id = 0
    _removed_dup_reviews = 0

print(f'df_reviews_tech shape: {df_reviews_tech.shape}')
print(f'So dong bi loai (khong co review_id) : {_dropped_no_id}')
print(f'So review trung lap da loai          : {_removed_dup_reviews}')


In [ ]:
# ── Technical cleaning: review_images ────────────────────────────────────────
df_images_tech = df_review_images_raw.copy()

if not df_images_tech.empty:
    _image_text_cols = ['restaurant_name', 'image_description', 'bg_color', 'photo_detail_url', 'image_url']
    for col in _image_text_cols:
        if col in df_images_tech.columns:
            df_images_tech[col] = df_images_tech[col].apply(technical_clean_text)

    if 'image_description' in df_images_tech.columns:
        df_images_tech['image_description'] = df_images_tech['image_description'].apply(
            lambda v: normalize_missing(v, fill='')
        )

    # Ep kieu numeric
    for col in ['image_id', 'review_id', 'restaurant_id', 'width', 'height', 'total_likes']:
        if col in df_images_tech.columns:
            df_images_tech[col] = pd.to_numeric(df_images_tech[col], errors='coerce')

    if 'total_likes' in df_images_tech.columns:
        df_images_tech['total_likes'] = df_images_tech['total_likes'].fillna(0).astype('Int64')

    if 'crawl_timestamp' in df_images_tech.columns:
        df_images_tech['crawl_timestamp'] = pd.to_datetime(
            df_images_tech['crawl_timestamp'], errors='coerce', utc=True
        )

    # Loai cac dong khong co image_url (khong dung duoc cho multimodal — KHONG download anh o day)
    # Luu y: CSV doc lai chuoi rong '' thanh NaN, nen phai loai ca NaN lan chuoi rong/whitespace
    _before = len(df_images_tech)
    if 'image_url' in df_images_tech.columns:
        _has_url = df_images_tech['image_url'].notna() & (df_images_tech['image_url'].astype(str).str.strip() != '')
        df_images_tech = df_images_tech[_has_url]
    _dropped_no_url = _before - len(df_images_tech)

    # Loai trung lap: uu tien image_id, fallback (review_id, image_url)
    _before = len(df_images_tech)
    if 'image_id' in df_images_tech.columns and df_images_tech['image_id'].notna().any():
        df_images_tech.drop_duplicates(subset=['image_id'], keep='first', inplace=True)
    else:
        df_images_tech.drop_duplicates(subset=['review_id', 'image_url'], keep='first', inplace=True)
    df_images_tech.reset_index(drop=True, inplace=True)
    _removed_dup_images = _before - len(df_images_tech)

    logger.info(f'Review images: loai {_dropped_no_url} dong rong image_url, '
                f'loai {_removed_dup_images} dong trung lap -> con {len(df_images_tech):,} dong')
else:
    _dropped_no_url = 0
    _removed_dup_images = 0

print(f'df_images_tech shape: {df_images_tech.shape}')
print(f'So dong bi loai (image_url rong) : {_dropped_no_url}')
print(f'So anh trung lap da loai         : {_removed_dup_images}')


## 6. Phase 2 — Content Cleaning

**Muc tieu:** DANH DAU (khong xoa) cac van de ve noi dung — viec loc se thuc hien o Phase 3.

Tao cac cot moi tren `df_reviews_content` (ban sao cua `df_reviews_tech`, cot `comment` goc duoc giu nguyen):

| Cot | Y nghia |
|-----|---------|
| `comment_clean` | `comment` da bo URL, chuan hoa khoang trang (KHONG ghi de `comment` goc) |
| `emoji_count` | so luong emoji trong `comment_clean` |
| `is_ad_or_spam` | `True` neu chua tu khoa quang cao / spam |
| `comment_length` | do dai `comment_clean` (so ky tu) |
| `word_count` | so tu trong `comment_clean` |
| `is_too_short` | `True` neu qua ngan (`comment_length` hoac `word_count` duoi nguong) |
| `is_valid_content` | `True` neu noi dung du dieu kien de train (khong rong, du dai, khong spam, khong toan emoji) |


In [ ]:
# ── Cac ham xu ly cho Phase 2 ─────────────────────────────────────────────────

def remove_urls(text: str) -> str:
    """Loai bo lien ket http(s)://, www., va link rut gon khoi text."""
    return URL_PATTERN.sub(' ', text)


def count_emojis(text: str) -> int:
    """Dem so ky tu emoji trong text."""
    return len(EMOJI_PATTERN.findall(text))


def strip_emojis(text: str) -> str:
    """Loai bo cac ky tu emoji (dung de kiem tra comment 'toan emoji')."""
    return EMOJI_PATTERN.sub('', text)


def detect_ad_or_spam(text: str) -> bool:
    """Tra ve True neu text chua bat ky tu khoa quang cao / spam nao (khong phan biet hoa-thuong)."""
    lowered = text.lower()
    return any(keyword in lowered for keyword in SPAM_KEYWORDS)


def build_comment_clean(raw_comment: Any) -> str:
    """Tao ban 'comment_clean' cho NLP: bo URL, chuan hoa whitespace. KHONG dung de ghi de 'comment' goc."""
    text = raw_comment if isinstance(raw_comment, str) else ''
    text = remove_urls(text)
    text = normalize_whitespace(text)
    return text


print('Da dinh nghia ham Phase 2: remove_urls, count_emojis, strip_emojis,')
print('  detect_ad_or_spam, build_comment_clean')


In [ ]:
# ── Ap dung content cleaning cho reviews ──────────────────────────────────────
df_reviews_content = df_reviews_tech.copy()

if not df_reviews_content.empty:
    # comment_clean — truong phai sinh; cot 'comment' goc duoc giu nguyen, khong xoa
    df_reviews_content['comment_clean'] = df_reviews_content['comment'].apply(build_comment_clean)

    # Dac trung emoji
    df_reviews_content['emoji_count'] = df_reviews_content['comment_clean'].apply(count_emojis)

    # Co quang cao / spam (chi DANH DAU — khong xoa o day)
    df_reviews_content['is_ad_or_spam'] = df_reviews_content['comment_clean'].apply(detect_ad_or_spam)

    # Dac trung do dai noi dung
    df_reviews_content['comment_length'] = df_reviews_content['comment_clean'].str.len()
    df_reviews_content['word_count'] = (
        df_reviews_content['comment_clean'].str.split().apply(len)
    )

    # Co review qua ngan
    df_reviews_content['is_too_short'] = (
        (df_reviews_content['comment_length'] < MIN_COMMENT_LENGTH) |
        (df_reviews_content['word_count'] < MIN_WORD_COUNT)
    )

    # Comment "toan emoji": con noi dung sau khi bo URL/whitespace nhung rong sau khi bo emoji
    _stripped = df_reviews_content['comment_clean'].apply(strip_emojis).str.strip()
    _emoji_only = (df_reviews_content['comment_clean'].str.strip() != '') & (_stripped == '')

    # Co noi dung hop le — gop tat ca tin hieu o tren
    df_reviews_content['is_valid_content'] = (
        (df_reviews_content['comment_clean'].str.strip() != '') &
        (~_emoji_only) &
        (df_reviews_content['comment_length'] >= MIN_COMMENT_LENGTH) &
        (df_reviews_content['word_count'] >= MIN_WORD_COUNT) &
        (~df_reviews_content['is_ad_or_spam'])
    )

    _n_spam       = int(df_reviews_content['is_ad_or_spam'].sum())
    _n_short      = int(df_reviews_content['is_too_short'].sum())
    _n_emoji_only = int(_emoji_only.sum())
    _n_valid      = int(df_reviews_content['is_valid_content'].sum())

    logger.info(f'Content cleaning: {_n_spam} danh dau spam/quang cao, {_n_short} qua ngan, '
                f'{_n_emoji_only} toan emoji, {_n_valid} noi dung hop le '
                f'(tren tong {len(df_reviews_content):,} review)')
else:
    _n_spam = _n_short = _n_emoji_only = _n_valid = 0

print(f'df_reviews_content shape : {df_reviews_content.shape}')
print(f'Danh dau spam/quang cao  : {_n_spam:,}')
print(f'Danh dau qua ngan        : {_n_short:,}')
print(f'Danh dau toan emoji      : {_n_emoji_only:,}')
print(f'Danh dau noi dung hop le : {_n_valid:,}')


## 7. Phase 3 — ML Cleaning

**Muc tieu:** tao cac dataset san sang huan luyen cho bai toan **Regression**
(`avg_rating` la target — KHONG tao `rating_label`, KHONG chuyen sang classification).

Cac buoc:
1. Loc review co `avg_rating` hop le trong khoang `[0, 10]`
2. Tao `df_reviews_ml` — review du dieu kien train (noi dung hop le + rating hop le + co day du khoa)
3. Tao `df_review_images_clean` — anh map dung voi cac review trong `df_reviews_ml` (giu nguyen mapping)
4. Tao `multimodal_reviews` — join review + image theo `review_id` (1 anh = 1 dong)
5. Tao `text_only_reviews` — dataset chi dung text (vi du cho XLM-R)


In [ ]:
# ── Buoc 1+2: Loc rating hop le + review du dieu kien train (df_reviews_ml) ───
# Target hoi quy = avg_rating -> GIU NGUYEN gia tri lien tuc, KHONG bucket thanh nhan/lop.

if not df_reviews_content.empty:
    _has_rating = df_reviews_content['avg_rating'].notna()
    _rating_in_range = (
        (df_reviews_content['avg_rating'] >= MIN_VALID_RATING) &
        (df_reviews_content['avg_rating'] <= MAX_VALID_RATING)
    )
    _valid_rating_mask = _has_rating & _rating_in_range
    _n_invalid_rating = int((~_valid_rating_mask).sum())

    _has_ids_mask = df_reviews_content['review_id'].notna() & df_reviews_content['restaurant_id'].notna()

    _ml_mask = (
        df_reviews_content['is_valid_content'] &
        _has_ids_mask &
        _valid_rating_mask
    )

    df_reviews_ml = df_reviews_content[_ml_mask].copy()
    df_reviews_ml.reset_index(drop=True, inplace=True)

    logger.info(f'Loc ML: {_n_invalid_rating} dong co avg_rating thieu/ngoai khoang bi loai; '
                f'giu lai {len(df_reviews_ml):,} / {len(df_reviews_content):,} review de train')
else:
    df_reviews_ml = pd.DataFrame()
    _n_invalid_rating = 0

print(f'df_reviews_ml shape              : {df_reviews_ml.shape}')
print(f'Rating thieu/ngoai khoang hop le : {_n_invalid_rating:,}')
if not df_reviews_ml.empty:
    print(f'Khoang gia tri target avg_rating : [{df_reviews_ml["avg_rating"].min():.2f}, '
          f'{df_reviews_ml["avg_rating"].max():.2f}]  (regression target — gia tri lien tuc, giu nguyen)')


In [ ]:
# ── Buoc 3: Giu nguyen mapping multimodal (df_review_images_clean) ────────────
if not df_images_tech.empty and not df_reviews_ml.empty:
    _valid_review_ids = set(df_reviews_ml['review_id'].dropna().unique())

    _img_mask = (
        df_images_tech['review_id'].isin(_valid_review_ids) &
        df_images_tech['image_url'].notna() &
        (df_images_tech['image_url'].astype(str).str.strip() != '')
    )
    df_review_images_clean = df_images_tech[_img_mask].copy()
    df_review_images_clean.reset_index(drop=True, inplace=True)

    logger.info(f'Mapping multimodal: {len(df_review_images_clean):,} / {len(df_images_tech):,} '
                f'anh khop voi mot review_id hop le (co the dung de train)')
else:
    df_review_images_clean = pd.DataFrame()

print(f'df_review_images_clean shape : {df_review_images_clean.shape}')
if not df_review_images_clean.empty:
    print(f'So review co anh (unique)    : {df_review_images_clean["review_id"].nunique():,}')


In [ ]:
# ── Buoc 4: Dataset multimodal — moi dong la mot cap (review, image) ──────────
_multimodal_cols = [
    'review_id', 'restaurant_id', 'restaurant_name', 'comment_clean', 'avg_rating',
    'image_id', 'image_url', 'width', 'height', 'created_datetime',
]

if not df_review_images_clean.empty:
    _review_cols = [
        'review_id', 'restaurant_id', 'restaurant_name',
        'comment_clean', 'avg_rating', 'created_datetime',
    ]
    _review_subset = df_reviews_ml[[c for c in _review_cols if c in df_reviews_ml.columns]].copy()

    _image_cols = ['review_id', 'image_id', 'image_url', 'width', 'height']
    _image_subset = df_review_images_clean[[c for c in _image_cols if c in df_review_images_clean.columns]].copy()

    # Inner join theo review_id -> giu nguyen mapping 1 anh = 1 dong, khong mat anh / khong sinh review gia
    multimodal_reviews = _image_subset.merge(_review_subset, on='review_id', how='inner')
    multimodal_reviews = multimodal_reviews[[c for c in _multimodal_cols if c in multimodal_reviews.columns]]
    multimodal_reviews.reset_index(drop=True, inplace=True)

    logger.info(f'multimodal_reviews: {len(multimodal_reviews):,} dong '
                f'({multimodal_reviews["review_id"].nunique():,} review duy nhat)')
else:
    multimodal_reviews = pd.DataFrame(columns=_multimodal_cols)

print(f'multimodal_reviews shape : {multimodal_reviews.shape}')
if not multimodal_reviews.empty:
    # Kiem tra: 1 review co N anh phai sinh dung N dong trong multimodal_reviews
    _check = multimodal_reviews.groupby('review_id').size().sort_index()
    _expected = df_review_images_clean.groupby('review_id').size().sort_index()
    _mismatch = int((_check != _expected).sum())
    print(f'Kiem tra mapping 1-anh-1-dong : {"OK" if _mismatch == 0 else f"LECH o {_mismatch} review"}')


In [ ]:
# ── Buoc 5: Dataset chi-text (vi du cho mo hinh kieu XLM-R) ───────────────────
_text_only_cols = ['review_id', 'restaurant_id', 'comment_clean', 'avg_rating', 'created_datetime']

if not df_reviews_ml.empty:
    text_only_reviews = df_reviews_ml[[c for c in _text_only_cols if c in df_reviews_ml.columns]].copy()
    text_only_reviews.reset_index(drop=True, inplace=True)
else:
    text_only_reviews = pd.DataFrame(columns=_text_only_cols)

print(f'text_only_reviews shape : {text_only_reviews.shape}')
print(f'Cac cot                 : {list(text_only_reviews.columns)}')


## 8. Dataset Validation

Tong hop so lieu truoc/sau moi phase, phan phoi `avg_rating` (target hoi quy),
va do phu anh (image coverage) cho bai toan multimodal.


In [ ]:
# ── Bao cao validation ────────────────────────────────────────────────────────
print('=' * 65)
print('  BAO CAO VALIDATION DATASET')
print('=' * 65)

print('\n-- So luong raw --')
print(f'  Restaurants raw     : {len(df_restaurants_raw):,}')
print(f'  Reviews raw         : {len(df_reviews_raw):,}')
print(f'  Review images raw   : {len(df_review_images_raw):,}')

print('\n-- Sau Phase 1 (Technical Cleaning) --')
print(f'  Restaurants         : {len(df_restaurants_tech):,}')
print(f'  Reviews             : {len(df_reviews_tech):,}')
print(f'  Review images       : {len(df_images_tech):,}')

print('\n-- Sau Phase 2 (Content Cleaning — chi danh dau) --')
print(f'  Reviews (da gan co) : {len(df_reviews_content):,}')
print(f'    is_ad_or_spam     : {_n_spam:,}')
print(f'    is_too_short      : {_n_short:,}')
print(f'    toan emoji        : {_n_emoji_only:,}')
print(f'    is_valid_content  : {_n_valid:,}')

print('\n-- Sau Phase 3 (ML Filtering) --')
print(f'  Review hop le (df_reviews_ml)        : {len(df_reviews_ml):,}')
print(f'  Anh hop le (df_review_images_clean)  : {len(df_review_images_clean):,}')
print(f'  Cap multimodal (review+image)        : {len(multimodal_reviews):,}')
print(f'  Review chi-text (text_only_reviews)  : {len(text_only_reviews):,}')

print('\n-- So luong da loai bo --')
_removed_duplicates = _removed_dup_restaurants + _removed_dup_reviews + _removed_dup_images
print(f'  Trung lap (tat ca dataset)   : {_removed_duplicates:,}')
print(f'    restaurants : {_removed_dup_restaurants:,}')
print(f'    reviews     : {_removed_dup_reviews:,}')
print(f'    images      : {_removed_dup_images:,}')
print(f'  Review qua ngan (da danh dau)        : {_n_short:,}')
print(f'  Review spam/quang cao (da danh dau)  : {_n_spam:,}')
print(f'  Rating thieu / ngoai khoang hop le   : {_n_invalid_rating:,}')

print('\n-- Missing values trong df_reviews_ml --')
if not df_reviews_ml.empty:
    _missing = df_reviews_ml.isnull().sum()
    _missing = _missing[_missing > 0]
    if not _missing.empty:
        print(_missing.to_string())
    else:
        print('  (khong co gia tri thieu)')
else:
    print('  (df_reviews_ml rong)')

print('\n-- Phan phoi avg_rating (target hoi quy) --')
if not df_reviews_ml.empty:
    _ratings = df_reviews_ml['avg_rating'].dropna()
    print(f'  Mean   : {_ratings.mean():.3f}')
    print(f'  Median : {_ratings.median():.3f}')
    print(f'  Std    : {_ratings.std():.3f}')
    print(f'  Min    : {_ratings.min():.3f}')
    print(f'  Max    : {_ratings.max():.3f}')

    _bins   = [0, 2, 4, 6, 8, 10]
    _labels = ['0-2', '2-4', '4-6', '6-8', '8-10']
    _binned = pd.cut(_ratings, bins=_bins, labels=_labels, include_lowest=True)
    print('\n  Phan nhom rating (CHI de quan sat phan phoi — KHONG dung lam nhan train):')
    print(_binned.value_counts().sort_index().to_string())
else:
    print('  (khong co rating hop le de thong ke)')

print('\n-- Do phu anh (image coverage) --')
if not df_reviews_ml.empty:
    _n_with_images = df_review_images_clean['review_id'].nunique() if not df_review_images_clean.empty else 0
    _pct = _n_with_images / len(df_reviews_ml) * 100 if len(df_reviews_ml) else 0.0
    print(f'  Review co >= 1 anh hop le : {_n_with_images:,} / {len(df_reviews_ml):,} ({_pct:.1f}%)')
    print(f'  Tong so cap multimodal    : {len(multimodal_reviews):,}')
else:
    print('  (khong co review hop le de danh gia do phu anh)')

print('\n' + '=' * 65)


In [ ]:
# ── Bieu do phan phoi avg_rating ──────────────────────────────────────────────
if not df_reviews_ml.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df_reviews_ml['avg_rating'].dropna(), bins=20, color='#4C72B0', edgecolor='white')
    ax.set_title('Phan phoi avg_rating (Regression Target)')
    ax.set_xlabel('avg_rating')
    ax.set_ylabel('So luong review')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('df_reviews_ml rong — bo qua ve bieu do.')


## 9. Save Cleaned Datasets

Luu toan bo output vao `checkpoints_clean/` (KHONG ghi de raw trong `checkpoints/`).
CSV (`utf-8-sig`) la dinh dang chinh; JSON la ban bo sung cho cac dataset chinh.


In [ ]:
# ── Luu dataset da lam sach — CSV la dinh dang chinh, JSON la ban bo sung ─────

def save_csv_json(df: pd.DataFrame, name: str, also_json: bool = True) -> Optional[str]:
    """Luu DataFrame thanh <name>.csv (utf-8-sig) va tuy chon <name>.json. Bo qua neu DataFrame rong."""
    if df.empty:
        logger.warning(f'{name}: DataFrame rong — bo qua khong luu.')
        return None

    csv_path = os.path.join(CLEAN_DIR, f'{name}.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    logger.info(f'Da luu: {csv_path} ({len(df):,} dong)')

    if also_json:
        json_path = os.path.join(CLEAN_DIR, f'{name}.json')
        df.to_json(json_path, orient='records', force_ascii=False, indent=2, date_format='iso')
        logger.info(f'Da luu: {json_path}')

    return csv_path


print('Dang luu cac dataset da lam sach vao:', CLEAN_DIR)
print('-' * 60)
save_csv_json(df_restaurants_tech,    'restaurants_clean')
save_csv_json(df_reviews_ml,          'reviews_clean')
save_csv_json(df_review_images_clean, 'review_images_clean')
save_csv_json(text_only_reviews,      'text_only_reviews',  also_json=False)
save_csv_json(multimodal_reviews,     'multimodal_reviews', also_json=False)
print('-' * 60)
print('Da luu xong cac file dataset.')


In [ ]:
# ── Tao va luu cleaning_report.json ──────────────────────────────────────────
cleaning_report: Dict[str, Any] = {
    'generated_at': pd.Timestamp.now(tz='UTC').isoformat(),
    'input_dir':  RAW_DIR,
    'output_dir': CLEAN_DIR,
    'task_type': 'regression',
    'target_column': 'avg_rating',
    'raw_counts': {
        'restaurants':   int(len(df_restaurants_raw)),
        'reviews':       int(len(df_reviews_raw)),
        'review_images': int(len(df_review_images_raw)),
    },
    'after_technical_cleaning': {
        'restaurants':   int(len(df_restaurants_tech)),
        'reviews':       int(len(df_reviews_tech)),
        'review_images': int(len(df_images_tech)),
    },
    'after_content_cleaning': {
        'reviews':               int(len(df_reviews_content)),
        'flagged_ad_or_spam':    int(_n_spam),
        'flagged_too_short':     int(_n_short),
        'flagged_emoji_only':    int(_n_emoji_only),
        'flagged_valid_content': int(_n_valid),
    },
    'after_ml_filtering': {
        'valid_reviews':                  int(len(df_reviews_ml)),
        'valid_review_images':            int(len(df_review_images_clean)),
        'multimodal_pairs':               int(len(multimodal_reviews)),
        'text_only_reviews':              int(len(text_only_reviews)),
        'invalid_or_out_of_range_ratings': int(_n_invalid_rating),
    },
    'removed': {
        'duplicate_restaurants': int(_removed_dup_restaurants),
        'duplicate_reviews':     int(_removed_dup_reviews),
        'duplicate_images':      int(_removed_dup_images),
        'reviews_no_review_id':  int(_dropped_no_id),
        'images_empty_url':      int(_dropped_no_url),
    },
    'rating_distribution': {},
    'image_coverage': {},
}

if not df_reviews_ml.empty:
    _ratings = df_reviews_ml['avg_rating'].dropna()
    cleaning_report['rating_distribution'] = {
        'mean':   float(_ratings.mean()),
        'median': float(_ratings.median()),
        'std':    float(_ratings.std()),
        'min':    float(_ratings.min()),
        'max':    float(_ratings.max()),
        'buckets': {
            str(k): int(v) for k, v in
            pd.cut(_ratings, bins=[0, 2, 4, 6, 8, 10],
                   labels=['0-2', '2-4', '4-6', '6-8', '8-10'],
                   include_lowest=True).value_counts().sort_index().items()
        },
    }
    _n_with_images = df_review_images_clean['review_id'].nunique() if not df_review_images_clean.empty else 0
    cleaning_report['image_coverage'] = {
        'reviews_with_images': int(_n_with_images),
        'total_valid_reviews': int(len(df_reviews_ml)),
        'coverage_pct': round(_n_with_images / len(df_reviews_ml) * 100, 2) if len(df_reviews_ml) else 0.0,
    }

report_path = os.path.join(CLEAN_DIR, 'cleaning_report.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(cleaning_report, f, ensure_ascii=False, indent=2)

print(f'Da luu: {report_path}')
print()
print(json.dumps(cleaning_report, ensure_ascii=False, indent=2))


## 10. Final Report

Tong ket toan bo pipeline cleaning va liet ke cac file da luu trong `checkpoints_clean/`.


In [ ]:
# ── Final Report ─────────────────────────────────────────────────────────────
print('FOODY DATASET CLEANING — HOAN TAT')
print('=' * 55)
print(f'Input dir   : {RAW_DIR}')
print(f'Output dir  : {CLEAN_DIR}')
print(f'Task type   : Regression  (target = avg_rating, gia tri lien tuc)')
print()
print(f'Restaurants (clean)        : {len(df_restaurants_tech):,}')
print(f'Reviews (clean, ML-ready)  : {len(df_reviews_ml):,}')
print(f'Review images (clean)      : {len(df_review_images_clean):,}')
print(f'Multimodal pairs           : {len(multimodal_reviews):,}')
print(f'Text-only reviews          : {len(text_only_reviews):,}')
print()

print('Cac file da luu:')
try:
    found = False
    for fname in sorted(os.listdir(CLEAN_DIR)):
        fpath = os.path.join(CLEAN_DIR, fname)
        if os.path.isfile(fpath):
            size_kb = os.path.getsize(fpath) / 1024
            print(f'  {fname:<35}  ({size_kb:,.1f} KB)')
            found = True
    if not found:
        print(f'  (khong tim thay file nao trong {CLEAN_DIR})')
except OSError as e:
    print(f'  Khong the liet ke thu muc output: {e}')

print()
print('Tom tat pipeline:')
print('  Phase 1 (Technical) -> giai ma HTML entity, chuan hoa Unicode/whitespace,')
print('                          parse /Date(...)/ , ep kieu numeric, loai trung lap')
print('  Phase 2 (Content)   -> sinh comment_clean, danh dau spam/qua ngan/toan emoji')
print('  Phase 3 (ML)        -> loc rating + noi dung hop le, dung dataset multimodal')
print('                          va text-only de train cho bai toan Regression')
print('=' * 55)
